# Explore RSNA Pneumonia

This notebook uses the standard PyHealth flow: RSNA dataset → `set_task()` → `SampleDataset` → `get_dataloader()`. `dev=True` limits exploration to 1000 patients. RSNA has no segmentation mask; `overlay_mask` is accepted for helper consistency. RSNA requires a manual Kaggle download; set `PYHEALTH_IMAGING_DATA` to the folder containing `RSNAPneumoniaDetection2018`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyhealth.datasets import RSNAPneumoniaDataset, get_dataloader

DATA_ROOT = Path(
    os.environ.get("PYHEALTH_IMAGING_DATA", "~/pyhealth-imaging-data")
).expanduser()
# RSNA requires a manual Kaggle download. Download it first, then place
# RSNAPneumoniaDetection2018 under DATA_ROOT.
CACHE_DIR = Path(
    os.environ.get("PYHEALTH_IMAGING_CACHE", "~/.cache/pyhealth-imaging")
).expanduser()
# Processed caches and generated metadata are stored here, outside the repo.
METADATA_DIR = Path(CACHE_DIR) / "metadata"

dataset = RSNAPneumoniaDataset(
    root=DATA_ROOT / "RSNAPneumoniaDetection2018",
    cache_dir=CACHE_DIR,
    metadata_dir=str(METADATA_DIR / "rsna_pneumonia"),
    dev=True,
)
dataset.stats()


def show_rsna_samples(samples, indices, overlay_mask=False):
    fig, axes = plt.subplots(
        1, len(indices), figsize=(4 * len(indices), 4), squeeze=False
    )
    for col, index in enumerate(indices):
        sample = samples[index]
        image = sample["image"][0].cpu()
        axes[0, col].imshow(image, cmap="gray")
        if overlay_mask and "mask" in sample:
            mask = sample["mask"][0].cpu()
            axes[0, col].imshow(
                np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35
            )
        axes[0, col].set_title(
            f"{sample['patient_id'][:8]}... label={int(sample['pneumonia'])}"
        )
        axes[0, col].axis("off")
    plt.show()


In [ ]:
patient_ids = dataset.unique_patient_ids
print(patient_ids[:5])
dataset.get_patient(patient_ids[0]).get_events()[0].attr_dict


In [ ]:
task = dataset.default_task
print(task.task_name)
print(task.input_schema)
print(task.output_schema)


In [ ]:
samples = dataset.set_task(task, num_workers=1)
print("samples:", len(samples))
print("input schema:", samples.input_schema)
print("output schema:", samples.output_schema)


In [ ]:
sample = samples[0]
print(sample.keys())
print("image:", sample["image"].shape, sample["image"].dtype)
print("pneumonia:", sample["pneumonia"])


In [ ]:
loader = get_dataloader(samples, batch_size=2)
batch = next(iter(loader))
print("batch image:", batch["image"].shape)
print("batch pneumonia:", batch["pneumonia"].shape)


In [ ]:
labels = [int(samples[i]["pneumonia"]) for i in range(len(samples))]
positive = labels.index(1)
negative = labels.index(0)
indices = [positive, negative]
for index, label in enumerate(labels):
    if len(indices) >= 3:
        break
    if index not in indices and label != labels[indices[0]]:
        indices.append(index)

show_rsna_samples(samples, indices, overlay_mask=False)
